# AL-DIDAH — تحليل كامل الإطارات وإعداد الإنتاج v2

يعتمد هذا الدفتر على محرك موجود داخل مستودع GitHub. يعالج كل إطار من الفيديو، يربط الكشوف بمعرّفات تتبع ثابتة داخل اللقطة، ويحسب السرعات فقط عندما تكون القياسات متجاورة. يحافظ على مقدمة الشعار، ويصدر البيانات والمعاينات إلى GitHub Release.

لا يستعيد MP4 الكود الأصلي أو العمق ثلاثي الأبعاد المخفي. النتائج بيانات هندسة عكسية قابلة للمراجعة، وليست حلقة أنمي نهائية أو Rig جاهزًا.


## 1 — تهيئة GPU ورمز النشر
اختر Runtime → Change runtime type → T4 GPU. أضف GitHub PAT إلى Colab Secrets باسم `GH_TOKEN` مع Contents: Read and write للمستودع.


In [ ]:
from google.colab import userdata
import os, subprocess, sys, pathlib, torch
assert torch.cuda.is_available(), 'فعّل T4 GPU من إعدادات Runtime.'
print('GPU:', torch.cuda.get_device_name(0))
GH_TOKEN = userdata.get('GH_TOKEN')
assert GH_TOKEN, 'أضف GitHub Secret باسم GH_TOKEN.'
os.environ['GH_TOKEN'] = GH_TOKEN


## 2 — استنساخ المستودع وتشغيل خط الإنتاج
المعالجة تحفظ الملفات محليًا في Colab ثم تنشئ إصدارًا جديدًا في GitHub Releases، دون الحاجة إلى تركيب Drive.


In [ ]:
REPO_DIR = pathlib.Path('/content/AI-Creative-Studio')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/h776117073-eng/AI-Creative-Studio.git',str(REPO_DIR)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.0','scenedetect[opencv]>=0.6.4','gdown>=5.2','rembg[cpu]>=2.0.60','onnxruntime>=1.19','pandas>=2.2','scipy>=1.13','Pillow>=10','requests>=2.32','tqdm>=4.66'],check=True)
subprocess.run([sys.executable,str(REPO_DIR/'tasks'/'current_task.py')],cwd=str(REPO_DIR),env=os.environ.copy(),check=True)


## 3 — فحص GitHub Release بعد التشغيل


In [ ]:
import requests
u='https://api.github.com/repos/h776117073-eng/AI-Creative-Studio/releases/tags/al-didah-production-v2'
r=requests.get(u,headers={'Authorization':f'Bearer {GH_TOKEN}','Accept':'application/vnd.github+json'},timeout=40)
r.raise_for_status(); rel=r.json()
print(rel['html_url'])
for a in rel.get('assets',[]): print(a['name'],a['size'],a['browser_download_url'])
